# Step 3 · FNO — learn the solution OPERATOR

A space-time Fourier Neural Operator maps any initial condition $u_0(x)$ to the
full field $u(x,t)$ in one forward pass (~3 ms). With **no x-coordinate channel**
it is exactly translation-equivariant, and because its spectral weights are indexed
by mode number it is resolution-invariant (zero-shot super-resolution). Thin wrapper
over `shocklab`; to (re)train: `python -m shocklab.train.fno --config configs/fno.yaml`.

In [ ]:
import torch

from shocklab.train.common import get_device
from shocklab.train.fno import FNOConfig, build_model, evaluate

cfg = FNOConfig.from_yaml("configs/fno.yaml")
device = get_device("cuda")
model = build_model(cfg).to(device).eval()
model.load_state_dict(torch.load("checkpoints/fno/model.pt", map_location=device))
print(f"FNO: {model.num_params():,} params (modes {cfg.modes_x}x{cfg.modes_t}, width {cfg.width})")

In [ ]:
# test @256, zero-shot @1024, zero-shot in t (classic), and latency
res = evaluate(model, device)
for k, v in res.items():
    print(k, v)

In [ ]:
# quick equivariance check: shifting u0 shifts the output (to float tolerance)
from shocklab.data.datasets import load_test, subsample_x

test = load_test()
u0 = torch.as_tensor(subsample_x(test["u"], 4)[:1, 0, :], device=device, dtype=torch.float32)
t = torch.linspace(0, 1, 51, device=device)
with torch.no_grad():
    y = model(u0, t)
    y_shift = model(torch.roll(u0, 20, dims=1), t)
err = (y_shift - torch.roll(y, 20, dims=1)).abs().max() / y.abs().max()
print(f"translation-equivariance error: {err.item():.2e}")

In [ ]:
from IPython.display import Image

# trained at 256, evaluated zero-shot at 1024 — the shock resolved more finely
Image(filename="checkpoints/fno/figures/superres.png")

**Takeaway.** One model, every initial condition, ~0.5 % error, ~3 ms, and the same
error at 1024 as at 256. Step 4 adds the physics back in.